# Chapter 3 - Implementing Video Generation Pipelines

Companion notebook for *Video Generation with AI* (O'Reilly) by Joseph Enochs.

**Learning objective:** Build intuition for working with video data and learn how to implement staged, modular training pipelines for video generation with AI.

This notebook implements the core components of a Latte-style (Latent Diffusion Transformer) video generation model, in the same order as the chapter: VAE latent compression, video tokenization, patch embedding with dual positional encoding, spatial and temporal attention, diffusion training, and text conditioning. Everything runs top to bottom on a free Colab CPU or GPU using a tiny synthetic video that the notebook creates itself.

In [ ]:
# Install dependencies (Colab already ships with torch and matplotlib)
%pip install -q numpy matplotlib tqdm "transformers>=4.40" sentencepiece

## Setup

Imports, a fixed random seed for reproducibility, and device selection. Everything below is sized to run on CPU in a few minutes; a GPU makes it faster but is not required.

In [ ]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Building the Foundation: Data Pipeline and Encoding

Raw video is too massive and redundant to model directly, so every video generation model starts by compressing, encoding, and structuring the data. We first need a video to work with, so we create a tiny synthetic clip: a colored square sliding across the frame. It is small enough to run everywhere but still has real spatial structure and real motion.

In [ ]:
def make_moving_square_video(num_frames=8, size=64, square=12):
    """Create a small synthetic video of a square moving across the frame."""
    frames = np.zeros((num_frames, size, size, 3), dtype=np.float32)
    for t in range(num_frames):
        x0 = int((size - square) * t / max(num_frames - 1, 1))
        y0 = size // 2 - square // 2
        frames[t, y0:y0 + square, x0:x0 + square, 0] = 1.0
        frames[t, y0:y0 + square, x0:x0 + square, 1] = 0.4
    return frames


NUM_FRAMES, FRAME_SIZE = 8, 64
video_np = make_moving_square_video(NUM_FRAMES, FRAME_SIZE)

# To tensor: [batch, channels, frames, height, width]
video = torch.from_numpy(video_np).permute(3, 0, 1, 2).unsqueeze(0).to(device)
print('Video tensor shape:', tuple(video.shape))

fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(2 * NUM_FRAMES, 2))
for t, ax in enumerate(axes):
    ax.imshow(video_np[t])
    ax.set_title(f'frame {t}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Latent Space Compression with VAE

The VAE acts as a learned compression engine: all generative modeling happens in its compressed latent space, cutting compute by one to two orders of magnitude. Video VAEs use 3D convolutions with carefully chosen kernels:

- `(1, 3, 3)` kernels extract spatial features while keeping frames temporally independent
- `(1, 4, 4)` kernels with stride `(1, 2, 2)` downsample space while preserving temporal resolution

The encoder below progressively downsamples 8x in space, projects to a latent distribution (`mu`, `logvar`), and uses the reparameterization trick so gradients can flow through sampling. The `encode_to_latent` method is the deterministic inference path that returns only the mean.

In [ ]:
# VAE Encoder implementation for video compression
class VideoVAEEncoder(nn.Module):
    def __init__(self, in_channels=3,
                 latent_channels=4, base_channels=128):
        super().__init__()
        # Progressive downsampling through 3D convolutions
        # kernel_size=(1, h, w): the leading 1 preserves time!
        self.conv1 = nn.Conv3d(
            in_channels, base_channels,
            kernel_size=(1, 3, 3),
            stride=(1, 1, 1),
            padding=(0, 1, 1)
        )
        self.conv2 = nn.Conv3d(
            base_channels, base_channels * 2,
            kernel_size=(1, 4, 4),
            stride=(1, 2, 2),  # 2x downsample
            padding=(0, 1, 1)
        )
        self.conv3 = nn.Conv3d(
            base_channels * 2, base_channels * 4,
            kernel_size=(1, 4, 4),
            stride=(1, 2, 2),  # 4x total
            padding=(0, 1, 1)
        )
        self.conv4 = nn.Conv3d(
            base_channels * 4, base_channels * 4,
            kernel_size=(1, 4, 4),
            stride=(1, 2, 2),  # 8x total
            padding=(0, 1, 1)
        )

        # Project to latent distribution parameters
        self.conv_mu = nn.Conv3d(
            base_channels * 4, latent_channels,
            kernel_size=(1, 3, 3), padding=(0, 1, 1)
        )
        self.conv_logvar = nn.Conv3d(
            base_channels * 4, latent_channels,
            kernel_size=(1, 3, 3), padding=(0, 1, 1)
        )

        # Normalization for training stability
        self.norm = nn.GroupNorm(
            num_groups=32, num_channels=base_channels * 4
        )

    def forward(self, x):
        # x shape: [batch, channels, frames, height, width]

        # Encode through convolutional layers
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))  # 2x downsample
        h = F.relu(self.conv3(h))  # 4x downsample
        h = F.relu(self.conv4(h))  # 8x downsample
        h = self.norm(h)

        # Get distribution parameters
        mu = self.conv_mu(h)
        logvar = self.conv_logvar(h)

        # Reparameterization trick for sampling
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        return z, mu, logvar

    def encode_to_latent(self, x):
        # For inference, just return the mean
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        h = F.relu(self.conv4(h))
        h = self.norm(h)
        mu = self.conv_mu(h)
        return mu

We instantiate a small version (`base_channels=32` instead of 128) and encode our clip. A 64 x 64 frame becomes an 8 x 8 latent "sketch". Note that Latte itself reuses a pretrained Stable Diffusion VAE rather than training its own; this implementation is for building intuition about what happens inside.

In [ ]:
vae_encoder = VideoVAEEncoder(
    in_channels=3, latent_channels=4, base_channels=32
).to(device)

with torch.no_grad():
    z, mu, logvar = vae_encoder(video)
    latent = vae_encoder.encode_to_latent(video)

print('Input video:          ', tuple(video.shape))
print('Sampled latent z:     ', tuple(z.shape))
print('Deterministic latent: ', tuple(latent.shape))

pixel_values = video.numel()
latent_values = latent.numel()
print(f'Compression: {pixel_values:,} values -> {latent_values:,} values '
      f'({pixel_values / latent_values:.0f}x smaller)')

## 2. Video Tokenization: From Pixels to Patches

Transformers process discrete token sequences, so we divide each latent frame into non-overlapping patches. At book scale, a 256 x 256 frame compresses to a 32 x 32 x 4 latent, and 2 x 2 patches give 256 tokens per frame, so 4,096 tokens for a 16-frame clip. Patch size is a key trade-off: smaller patches keep detail but lengthen the sequence; larger patches shorten it but lose fine structure.

In [ ]:
def count_tokens(latent_hw, patch_size, num_frames):
    patches_per_frame = (latent_hw // patch_size) ** 2
    return patches_per_frame, patches_per_frame * num_frames


# Book scale: 256 x 256 video -> 32 x 32 latent, 16 frames
ppf, total = count_tokens(32, 2, 16)
print(f'Book scale: {ppf} patches per frame, {total} total tokens')

# Our toy scale: 64 x 64 video -> 8 x 8 latent, 8 frames
LATENT_SIZE = FRAME_SIZE // 8
PATCH_SIZE = 2
ppf, total = count_tokens(LATENT_SIZE, PATCH_SIZE, NUM_FRAMES)
print(f'Toy scale:  {ppf} patches per frame, {total} total tokens')

# Effect of patch size on sequence length (book scale)
for p in [1, 2, 4, 8]:
    _, t = count_tokens(32, p, 16)
    print(f'patch {p} x {p}: {t:,} tokens')

## 3. Patch Embedding and Positional Encoding

Each patch is flattened and projected into a high-dimensional embedding space, then two positional encodings are added: a spatial one (where in the frame) and a temporal one (which frame). Both are initialized with sinusoidal patterns, which give nearby positions similar signatures and generalize to unseen sequence lengths. Chapter 5 introduces RoPE as the modern alternative.

In [ ]:
# Patch embedding with dual positional encoding
def get_sincos_encoding(n_position, d_hid):
    """Sinusoidal encoding table: sin on even indices, cos on odd indices."""
    def get_position_angle_vec(position):
        return [
            position / np.power(10000, 2 * (hid_j // 2) / d_hid)
            for hid_j in range(d_hid)
        ]

    sincos_table = np.array([
        get_position_angle_vec(pos_i)
        for pos_i in range(n_position)
    ])

    # Apply sin to even indices
    sincos_table[:, 0::2] = np.sin(sincos_table[:, 0::2])
    # Apply cos to odd indices
    sincos_table[:, 1::2] = np.cos(sincos_table[:, 1::2])

    return torch.FloatTensor(sincos_table).unsqueeze(0)


class VideoPatchEmbed(nn.Module):
    def __init__(self, patch_size=2, in_channels=4,
                 embed_dim=768, num_frames=16,
                 image_size=32):
        super().__init__()
        self.patch_size = patch_size

        # Calculate total patches
        self.num_patches_per_frame = (
            (image_size // patch_size) ** 2
        )
        self.num_patches = (
            self.num_patches_per_frame * num_frames
        )

        # Linear projection: patches -> embeddings
        patch_dim = in_channels * patch_size * patch_size
        self.proj = nn.Linear(patch_dim, embed_dim)

        # Dual positional encodings
        self.spatial_pos_embed = nn.Parameter(
            torch.zeros(1,
                        self.num_patches_per_frame,
                        embed_dim)
        )
        self.temporal_pos_embed = nn.Parameter(
            torch.zeros(1, num_frames, embed_dim)
        )

        # Initialize with sinusoidal patterns
        self._init_pos_encoding()

    def _init_pos_encoding(self):
        # Initialize spatial positions
        self.spatial_pos_embed.data.copy_(
            get_sincos_encoding(
                self.num_patches_per_frame,
                self.spatial_pos_embed.shape[-1]
            )
        )

        # Initialize temporal positions
        self.temporal_pos_embed.data.copy_(
            get_sincos_encoding(
                self.temporal_pos_embed.shape[1],
                self.temporal_pos_embed.shape[-1]
            )
        )

    def forward(self, x):
        # x shape: [batch, channels, frames, height, width]
        B, C, T, H, W = x.shape
        assert H % self.patch_size == 0 and W % self.patch_size == 0

        # Reshape into patches
        x = x.reshape(B, C, T, H // self.patch_size, self.patch_size,
                      W // self.patch_size, self.patch_size)
        x = x.permute(0, 2, 3, 5, 4, 6, 1).reshape(
            B, T * (H // self.patch_size) * (W // self.patch_size),
            C * self.patch_size * self.patch_size
        )

        # Apply linear projection
        x = self.proj(x)  # [B, T*num_patches_per_frame, embed_dim]

        # Add dual positional encoding
        x = x.reshape(B, T, -1, x.shape[-1])

        # Add spatial position to each frame
        x = x + self.spatial_pos_embed.unsqueeze(1)

        # Add temporal position across frames
        x = x + self.temporal_pos_embed.unsqueeze(2)

        # Flatten back to sequence
        x = x.reshape(B, -1, x.shape[-1])

        return x

We embed our toy latents into a 128-dimensional token sequence and visualize the sinusoidal table. Each row (position) gets a unique smooth signature, and nearby rows look alike.

In [ ]:
EMBED_DIM = 128

patch_embed = VideoPatchEmbed(
    patch_size=PATCH_SIZE, in_channels=4, embed_dim=EMBED_DIM,
    num_frames=NUM_FRAMES, image_size=LATENT_SIZE,
).to(device)

with torch.no_grad():
    tokens = patch_embed(latent)

PATCHES_PER_FRAME = patch_embed.num_patches_per_frame
print('Latent video:   ', tuple(latent.shape))
print('Token sequence: ', tuple(tokens.shape))
print('Patches per frame:', PATCHES_PER_FRAME)

enc = get_sincos_encoding(64, EMBED_DIM)[0].numpy()
plt.figure(figsize=(8, 3))
plt.imshow(enc, aspect='auto', cmap='RdBu')
plt.xlabel('embedding dimension')
plt.ylabel('position')
plt.title('Sinusoidal positional encoding')
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Attention Mechanisms: Understanding Relationships in Space and Time

Latte separates the two core relationship problems of video. Spatial attention keeps each frame coherent (a cat's ears stay attached to its head), while temporal attention keeps motion consistent across frames (the cat walks smoothly instead of teleporting). Temporal attention groups patches by spatial position and attends across frames; an optional causal mask restricts each frame to look only backward, which matters for autoregressive generation.

In [ ]:
# Temporal attention block implementation
class TemporalAttentionBlock(nn.Module):
    """Temporal dimension - how frames relate to each other"""

    def __init__(self, dim, num_heads=8, num_frames=16,
                 num_patches_per_frame=256, causal=False):
        super().__init__()
        self.num_heads = num_heads
        self.num_frames = num_frames
        self.num_patches_per_frame = num_patches_per_frame
        self.causal = causal  # for autoregressive generation
        self.scale = (dim // num_heads) ** -0.5

        # Multi-head attention components
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

        # Create causal mask if needed
        if causal:
            self.register_buffer(
                "causal_mask",
                torch.tril(torch.ones(num_frames, num_frames))
            )

    def forward(self, x, text_embed=None):
        # x shape: [batch, num_patches_total, dim]
        B, N, C = x.shape

        # Reshape to separate temporal and spatial dimensions
        # Group patches by spatial position across frames
        x = x.reshape(B, self.num_frames, self.num_patches_per_frame, C)
        # [B, num_patches_per_frame, num_frames, C]
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(B * self.num_patches_per_frame, self.num_frames, C)

        # Apply layer norm and save for residual
        shortcut = x
        x = self.norm(x)

        # Compute Q, K, V
        qkv = self.qkv(x).reshape(
            B * self.num_patches_per_frame, self.num_frames,
            3, self.num_heads, C // self.num_heads
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Compute attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale

        # Apply causal mask if needed (for autoregressive generation)
        if self.causal:
            mask_value = -torch.finfo(attn.dtype).max
            mask = self.causal_mask[:self.num_frames, :self.num_frames]
            mask = mask.reshape(1, 1, self.num_frames, self.num_frames)
            attn = attn.masked_fill(mask == 0, mask_value)

        # Apply softmax and attention
        attn = F.softmax(attn, dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(
            B * self.num_patches_per_frame, self.num_frames, C
        )

        # Project and add residual
        x = self.proj(x)
        x = shortcut + x

        # Reshape back to original format
        x = x.reshape(B, self.num_patches_per_frame, self.num_frames, C)
        x = x.permute(0, 2, 1, 3)  # [B, num_frames, num_patches_per_frame, C]
        x = x.reshape(B, N, C)

        return x

Spatial attention is the complementary block: it reshapes so each frame is processed independently with standard multi-head self-attention. While temporal attention asks "how does this patch evolve over time?", spatial attention asks "how does this patch relate to other patches in the same frame?"

In [ ]:
class SpatialAttentionBlock(nn.Module):
    """Spatial dimension - how patches relate within a frame"""

    def __init__(self, dim, num_heads=8, num_frames=16):
        super().__init__()
        self.num_heads = num_heads
        self.num_frames = num_frames
        self.scale = (dim // num_heads) ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x shape: [batch, num_patches_total, dim]
        B, N, C = x.shape
        patches_per_frame = N // self.num_frames

        # Reshape to process each frame independently
        x = x.reshape(B * self.num_frames, patches_per_frame, C)

        # Standard multi-head self-attention
        shortcut = x
        x = self.norm(x)

        qkv = self.qkv(x).reshape(
            B * self.num_frames, patches_per_frame,
            3, self.num_heads, C // self.num_heads
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention computation
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(
            B * self.num_frames, patches_per_frame, C
        )
        x = self.proj(x)
        x = shortcut + x

        # Reshape back to full sequence
        x = x.reshape(B, N, C)
        return x

In [ ]:
NUM_HEADS = 8

spatial_attn = SpatialAttentionBlock(
    EMBED_DIM, NUM_HEADS, NUM_FRAMES
).to(device)
temporal_attn = TemporalAttentionBlock(
    EMBED_DIM, NUM_HEADS, NUM_FRAMES, PATCHES_PER_FRAME
).to(device)
causal_attn = TemporalAttentionBlock(
    EMBED_DIM, NUM_HEADS, NUM_FRAMES, PATCHES_PER_FRAME, causal=True
).to(device)

with torch.no_grad():
    out = spatial_attn(tokens)
    out = temporal_attn(out)
    out_causal = causal_attn(tokens)

print('After spatial + temporal attention:', tuple(out.shape))
print('With causal temporal attention:    ', tuple(out_causal.shape))
print('Causal mask (frames x frames), 1 = may attend:')
print(causal_attn.causal_mask.int().cpu().numpy())

### The Alternating Block Architecture

Rather than modeling all spatiotemporal relationships at once, Latte alternates spatial and temporal blocks. This drops the cost from O(T^2 x H^2 x W^2) for full 3D attention to O(T x H^2 x W^2 + T^2 x H x W), roughly the difference between 5 GB and 256 GB of attention memory at 256 x 256 resolution. Most successful implementations stack 12 to 24 alternating blocks; Table 3-1 in the chapter lists the Latte-S/B/L/XL variants.

In [ ]:
# Alternate spatial and temporal processing, the core Latte pattern
x = tokens
with torch.no_grad():
    for _ in range(2):
        x = spatial_attn(x)
        x = temporal_attn(x)
print('After 2 spatial/temporal alternations:', tuple(x.shape))

# Latte architecture variants (Table 3-1)
variants = [
    ('Latte-S', 12, '84M', 384, 6, 'fast prototyping'),
    ('Latte-B', 12, '151M', 768, 12, 'standard choice'),
    ('Latte-L', 24, '459M', 1024, 16, 'high quality'),
    ('Latte-XL', 28, '674M', 1152, 16, 'maximum quality'),
]
header = f"{'Model':<10}{'Blocks':<8}{'Params':<9}{'Hidden':<8}{'Heads':<7}Use case"
print()
print(header)
print('-' * len(header))
for name, blocks, params, hidden, heads, use in variants:
    print(f'{name:<10}{blocks:<8}{params:<9}{hidden:<8}{heads:<7}{use}')

## 5. The Diffusion Process: From Noise to Video

Diffusion teaches the model to generate by first teaching it to destroy. The forward process gradually adds Gaussian noise to clean latents over T timesteps until only static remains. The reverse process is learned: at each step the model predicts the noise that was added, and removing it step by step recovers a clean video. The scheduler below implements a minimal linear-beta schedule with the `add_noise` forward process and a basic DDPM reverse `step` (the chapter's training loop only needs `add_noise` and `num_timesteps`).

In [ ]:
class NoiseScheduler:
    """Minimal linear-beta schedule: forward noising + DDPM reverse step."""

    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

    def to(self, device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        return self

    def add_noise(self, clean, noise, timesteps):
        # Forward process: corrupt clean latents at the given timesteps
        sqrt_alpha_prod = self.alphas_cumprod[timesteps] ** 0.5
        sqrt_one_minus = (1.0 - self.alphas_cumprod[timesteps]) ** 0.5

        # Broadcast over [batch, channels, frames, height, width]
        while sqrt_alpha_prod.dim() < clean.dim():
            sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)
            sqrt_one_minus = sqrt_one_minus.unsqueeze(-1)

        return sqrt_alpha_prod * clean + sqrt_one_minus * noise

    @torch.no_grad()
    def step(self, noise_pred, t, x):
        # One DDPM reverse step at integer timestep t
        beta_t = self.betas[t]
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alphas_cumprod[t]

        coef = beta_t / (1.0 - alpha_bar_t) ** 0.5
        mean = (x - coef * noise_pred) / alpha_t ** 0.5

        if t > 0:
            mean = mean + (beta_t ** 0.5) * torch.randn_like(x)
        return mean

Here is the forward process on one frame of our clip, shown in pixel space for intuition, plus the schedule that controls how quickly signal gives way to noise. We use 200 timesteps so the later sampling demo stays fast on CPU.

In [ ]:
scheduler = NoiseScheduler(num_timesteps=200).to(device)

frame = video[:, :, 4]  # [1, 3, H, W]
steps_to_show = [0, 50, 100, 150, 199]
fig, axes = plt.subplots(1, len(steps_to_show),
                         figsize=(2.2 * len(steps_to_show), 2.4))
for ax, t in zip(axes, steps_to_show):
    noise = torch.randn_like(frame)
    noisy = scheduler.add_noise(
        frame, noise, torch.tensor([t], device=device)
    )
    img = noisy[0].permute(1, 2, 0).cpu().clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(f't = {t}')
    ax.axis('off')
plt.suptitle('Forward process: clean frame to pure noise')
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(scheduler.alphas_cumprod.cpu().numpy())
plt.xlabel('timestep')
plt.ylabel('cumulative signal fraction')
plt.title('Noise schedule')
plt.tight_layout()
plt.show()

### Noise Prediction and Scheduling

The training loop is deliberately simple: sample random timesteps, add the corresponding noise, have the model predict that noise, and minimize the mean squared error. We define the trainer now and run it in section 9 once the full model is assembled.

In [ ]:
# Diffusion training loop implementation
class DiffusionTrainer:
    def __init__(self, model, noise_scheduler, optimizer, device='cuda'):
        self.model = model
        self.noise_scheduler = noise_scheduler
        self.optimizer = optimizer
        self.device = device

    def train_step(self, batch):
        # batch contains: latent_video, text_embeddings
        latent_video = batch['latent_video'].to(self.device)
        text_embed = batch['text_embed'].to(self.device)
        batch_size = latent_video.shape[0]

        # Sample random timesteps for each video in batch
        timesteps = torch.randint(
            0, self.noise_scheduler.num_timesteps,
            (batch_size,), device=self.device
        ).long()

        # Sample noise to add to the latents
        noise = torch.randn_like(latent_video)

        # Add noise to the latent videos according to the noise schedule
        noisy_latents = self.noise_scheduler.add_noise(
            latent_video, noise, timesteps
        )

        # Predict the noise with the model
        noise_pred = self.model(
            noisy_latents,
            timesteps,
            encoder_hidden_states=text_embed
        )

        # Calculate MSE loss between predicted and actual noise
        loss = F.mse_loss(noise_pred, noise)

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

        self.optimizer.step()

        return loss.item()

## 6. Adaptive Conditioning with AdaLN

Adaptive layer normalization lets external signals (the diffusion timestep, and by extension text or class conditioning) modulate every layer's features. A small MLP maps the timestep embedding to per-layer scale (`gamma`) and shift (`beta`) parameters. Early in generation the model establishes global structure; late in generation it refines detail. AdaLN is how it knows which mode it is in. Production Latte uses AdaLN-Zero, which adds zero-initialized gates for extra training stability; the core mechanism is the same.

In [ ]:
class AdaLN(nn.Module):
    """Adaptive Layer Normalization for timestep conditioning"""
    def __init__(self, dim, timestep_dim=256):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)

        # MLP to generate scale and shift from timestep
        self.timestep_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(timestep_dim, dim * 2)
        )

    def forward(self, x, timestep_embed):
        # Generate scale (gamma) and shift (beta) from timestep
        adaln_params = self.timestep_mlp(timestep_embed)
        adaln_params = adaln_params.unsqueeze(1)  # Add sequence dimension
        gamma, beta = adaln_params.chunk(2, dim=-1)

        # Apply adaptive normalization
        x = self.norm(x)
        x = (1 + gamma) * x + beta

        return x

## 7. Conditioning and Control: From Text to Video

### Text Encoding with T5

Latte encodes prompts with a T5 encoder, chosen over image-focused encoders like CLIP for its grasp of compositional language and temporal structure ("first... then..."). Prompts become a sequence of embeddings that will guide generation through cross-attention.

**Warning: model download.** Setting `LOAD_T5 = True` downloads the t5-small encoder (about 250 MB) from the Hugging Face Hub. The default `False` uses fixed random stand-in embeddings with the same shape, so the notebook runs fully offline. Every cell below works with either path.

In [ ]:
LOAD_T5 = False  # set True to download t5-small and encode the real prompt

CONTEXT_DIM = 512  # width of the t5-small encoder
prompt = 'a red square sliding across a dark background'

if LOAD_T5:
    from transformers import T5EncoderModel, T5Tokenizer
    tokenizer = T5Tokenizer.from_pretrained('t5-small')
    text_encoder = T5EncoderModel.from_pretrained('t5-small').to(device).eval()
    with torch.no_grad():
        tok = tokenizer(
            prompt, return_tensors='pt', padding='max_length',
            max_length=16, truncation=True
        ).to(device)
        text_embed = text_encoder(**tok).last_hidden_state
    print('T5 text embeddings:', tuple(text_embed.shape))
else:
    g = torch.Generator().manual_seed(0)
    text_embed = torch.randn(1, 16, CONTEXT_DIM, generator=g).to(device)
    print('Stand-in text embeddings:', tuple(text_embed.shape))

### Cross-Attention Integration

Cross-attention bridges the two modalities asymmetrically: video patches form the queries, text embeddings provide the keys and values. Each patch selectively attends to the words relevant to it, which is how "cat" patches find the word cat.

In [ ]:
# Cross-attention implementation for text conditioning
class CrossAttention(nn.Module):
    def __init__(self, dim, context_dim=768, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5

        # Separate projections for Q (from video) and K,V (from text)
        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(context_dim, dim, bias=False)
        self.to_v = nn.Linear(context_dim, dim, bias=False)

        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.context_norm = nn.LayerNorm(context_dim)

    def forward(self, x, context):
        # x: video features [batch, num_patches, dim]
        # context: text embeddings [batch, seq_len, context_dim]

        B, N, C = x.shape

        # Normalize inputs
        shortcut = x
        x = self.norm(x)
        context = self.context_norm(context)

        # Create queries from video features
        q = self.to_q(x).reshape(B, N, self.num_heads, C // self.num_heads)
        q = q.permute(0, 2, 1, 3)  # [B, heads, N, head_dim]

        # Create keys and values from text embeddings
        k = self.to_k(context).reshape(B, -1, self.num_heads, C // self.num_heads)
        k = k.permute(0, 2, 1, 3)  # [B, heads, seq_len, head_dim]

        v = self.to_v(context).reshape(B, -1, self.num_heads, C // self.num_heads)
        v = v.permute(0, 2, 1, 3)  # [B, heads, seq_len, head_dim]

        # Compute attention scores: Q @ K^T
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        # Apply attention to values: attn @ V
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)

        # Project and add residual
        out = self.proj(out)
        out = shortcut + out

        return out

In [ ]:
cross_attn = CrossAttention(
    EMBED_DIM, context_dim=CONTEXT_DIM, num_heads=NUM_HEADS
).to(device)

with torch.no_grad():
    conditioned = cross_attn(tokens, text_embed)
print('Tokens after cross-attention:', tuple(conditioned.shape))

### Classifier-Free Guidance

CFG sharpens prompt adherence at inference time. The model runs twice, once with the prompt and once without, and the final prediction amplifies the difference: the unconditional prediction plus `w` times (conditional minus unconditional). It is an inference-time technique, not an architectural component.

In [ ]:
def classifier_free_guidance(noise_uncond, noise_text, guidance_scale=7.5):
    """Combine conditional and unconditional predictions."""
    return noise_uncond + guidance_scale * (noise_text - noise_uncond)


# A scalar example: watch the conditional signal get amplified
eps_uncond = torch.tensor([1.0])
eps_text = torch.tensor([2.0])
for w in [0.0, 1.0, 4.0, 7.5]:
    guided = classifier_free_guidance(eps_uncond, eps_text, w)
    print(f'w = {w:>4}: guided prediction = {guided.item():.1f}')

## 8. Latte Model Component Integration

### Timestep Embedding

The model needs to know where it is in the denoising schedule, and a raw integer is not expressive enough. We reuse the sinusoidal encoding strategy on timesteps and refine it with a small MLP: nearby timesteps get similar embeddings (smooth interpolation), and models can often sample with fewer steps than they trained with (extrapolation).

In [ ]:
class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding refined by a small MLP."""

    def __init__(self, dim=256):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, timesteps):
        # timesteps: [batch] integer tensor
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=timesteps.device, dtype=torch.float32)
            / half
        )
        args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)

### The Complete Transformer Block

One representative ordering: AdaLN modulation, spatial attention (visual coherence), cross-attention (semantic grounding), temporal attention (consistency across frames), then a feed-forward network (capacity). Compared to the chapter listing, this version threads through `num_patches_per_frame` and `timestep_dim` so the temporal attention and AdaLN submodules are sized correctly for any configuration.

In [ ]:
class LatteTransformerBlock(nn.Module):
    """Complete transformer block with spatial, temporal, and cross attention"""
    def __init__(self, dim, num_heads=8, num_frames=16,
                 num_patches_per_frame=256, context_dim=768,
                 timestep_dim=256, use_adaln=True):
        super().__init__()
        self.use_adaln = use_adaln

        # Attention layers
        self.spatial_attn = SpatialAttentionBlock(dim, num_heads, num_frames)
        self.temporal_attn = TemporalAttentionBlock(
            dim, num_heads, num_frames, num_patches_per_frame
        )
        self.cross_attn = CrossAttention(dim, context_dim, num_heads)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

        # AdaLN modulation (if enabled)
        if use_adaln:
            self.adaln = AdaLN(dim, timestep_dim)

    def forward(self, x, timestep_embed, text_embed):
        # Apply AdaLN modulation if enabled
        if self.use_adaln:
            x = self.adaln(x, timestep_embed)

        # Spatial attention within frames
        x = self.spatial_attn(x)

        # Cross-attention with text
        x = self.cross_attn(x, text_embed)

        # Temporal attention across frames
        x = self.temporal_attn(x)

        # Feed-forward network
        x = x + self.ffn(x)

        return x

In [ ]:
t_embedder = TimestepEmbedding(dim=256).to(device)
block = LatteTransformerBlock(
    dim=EMBED_DIM, num_heads=NUM_HEADS, num_frames=NUM_FRAMES,
    num_patches_per_frame=PATCHES_PER_FRAME, context_dim=CONTEXT_DIM,
).to(device)

timesteps = torch.tensor([10], device=device)
with torch.no_grad():
    t_emb = t_embedder(timesteps)
    out = block(tokens, t_emb, text_embed)
print('Timestep embedding:', tuple(t_emb.shape))
print('Block output:      ', tuple(out.shape))

## 9. Putting It All Together: The Generation Pipeline

The complete pipeline: (1) encode the prompt, (2) sample random latent noise, (3) iteratively denoise with spatial, temporal, and cross attention modulated by AdaLN, (4) decode the final latent with the VAE, (5) output video. First we assemble a small end-to-end noise-prediction model: patch embedding in, transformer blocks in the middle, and an unpatchify head that projects tokens back to latent video.

In [ ]:
class LatteVideoModel(nn.Module):
    """Small Latte-style noise-prediction model for latent video."""

    def __init__(self, latent_channels=4, image_size=32, patch_size=2,
                 num_frames=16, dim=768, depth=4, num_heads=8,
                 context_dim=768, timestep_dim=256):
        super().__init__()
        self.latent_channels = latent_channels
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_frames = num_frames

        self.patch_embed = VideoPatchEmbed(
            patch_size=patch_size,
            in_channels=latent_channels,
            embed_dim=dim,
            num_frames=num_frames,
            image_size=image_size,
        )
        self.time_embed = TimestepEmbedding(timestep_dim)

        num_patches_per_frame = self.patch_embed.num_patches_per_frame
        self.blocks = nn.ModuleList([
            LatteTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_frames=num_frames,
                num_patches_per_frame=num_patches_per_frame,
                context_dim=context_dim,
                timestep_dim=timestep_dim,
            )
            for _ in range(depth)
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.head = nn.Linear(
            dim, patch_size * patch_size * latent_channels
        )

    def unpatchify(self, x):
        # Invert the patch reshaping: token sequence -> latent video
        B = x.shape[0]
        p = self.patch_size
        T = self.num_frames
        hp = wp = self.image_size // p
        C = self.latent_channels

        x = x.reshape(B, T, hp, wp, p, p, C)
        x = x.permute(0, 6, 1, 2, 4, 3, 5)  # [B, C, T, hp, p1, wp, p2]
        x = x.reshape(B, C, T, hp * p, wp * p)
        return x

    def forward(self, x, timesteps, encoder_hidden_states):
        # x: noisy latents [batch, latent_channels, frames, height, width]
        t_emb = self.time_embed(timesteps)
        tokens = self.patch_embed(x)

        for block in self.blocks:
            tokens = block(tokens, t_emb, encoder_hidden_states)

        tokens = self.head(self.final_norm(tokens))
        return self.unpatchify(tokens)

Now we train for a handful of steps on our single synthetic clip. This will not produce a good generator (that takes large datasets and many GPU hours, the subject of Chapter 4), but the loss dropping confirms the whole pipeline learns end to end.

In [ ]:
model = LatteVideoModel(
    latent_channels=4, image_size=LATENT_SIZE, patch_size=PATCH_SIZE,
    num_frames=NUM_FRAMES, dim=EMBED_DIM, depth=2, num_heads=NUM_HEADS,
    context_dim=CONTEXT_DIM,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params / 1e6:.2f}M')

# Normalize the latents so the diffusion target is well scaled
latent_train = (latent - latent.mean()) / (latent.std() + 1e-6)
batch = {'latent_video': latent_train, 'text_embed': text_embed}

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
trainer = DiffusionTrainer(model, scheduler, optimizer, device=device)

losses = [trainer.train_step(batch) for _ in tqdm(range(40))]
print(f'Loss: {losses[0]:.3f} -> {losses[-1]:.3f}')

plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel('training step')
plt.ylabel('MSE loss')
plt.title('Toy diffusion training on one synthetic clip')
plt.tight_layout()
plt.show()

With a lightly warmed up model, generation runs the reverse process: start from pure noise and repeatedly predict and remove noise, applying classifier-free guidance at each step.

In [ ]:
GUIDANCE_SCALE = 4.0
uncond_embed = torch.zeros_like(text_embed)

model.eval()
x = torch.randn_like(latent_train)
for t in tqdm(reversed(range(scheduler.num_timesteps)),
              total=scheduler.num_timesteps, desc='denoising'):
    ts = torch.tensor([t], device=device)
    with torch.no_grad():
        noise_text = model(x, ts, encoder_hidden_states=text_embed)
        noise_uncond = model(x, ts, encoder_hidden_states=uncond_embed)
    noise_pred = classifier_free_guidance(
        noise_uncond, noise_text, GUIDANCE_SCALE
    )
    x = scheduler.step(noise_pred, t, x)

generated_latent = x
print('Generated latent video:', tuple(generated_latent.shape))

The final pipeline step decodes latents back to pixels. The chapter's listings only implement the encoder (Latte reuses a pretrained Stable Diffusion VAE for both directions), so this compact decoder is a companion addition that mirrors the encoder's structure. Since neither the VAE nor the generator has been properly trained, expect abstract texture rather than a movie; the point is the shape round trip of the full pipeline.

In [ ]:
class VideoVAEDecoder(nn.Module):
    """Minimal mirror of VideoVAEEncoder (companion addition)."""

    def __init__(self, out_channels=3, latent_channels=4, base_channels=128):
        super().__init__()
        self.conv_in = nn.Conv3d(
            latent_channels, base_channels * 4,
            kernel_size=(1, 3, 3), padding=(0, 1, 1)
        )
        self.up1 = nn.ConvTranspose3d(
            base_channels * 4, base_channels * 4,
            kernel_size=(1, 4, 4), stride=(1, 2, 2), padding=(0, 1, 1)
        )
        self.up2 = nn.ConvTranspose3d(
            base_channels * 4, base_channels * 2,
            kernel_size=(1, 4, 4), stride=(1, 2, 2), padding=(0, 1, 1)
        )
        self.up3 = nn.ConvTranspose3d(
            base_channels * 2, base_channels,
            kernel_size=(1, 4, 4), stride=(1, 2, 2), padding=(0, 1, 1)
        )
        self.norm = nn.GroupNorm(num_groups=32, num_channels=base_channels)
        self.conv_out = nn.Conv3d(
            base_channels, out_channels,
            kernel_size=(1, 3, 3), padding=(0, 1, 1)
        )

    def forward(self, z):
        h = F.relu(self.conv_in(z))
        h = F.relu(self.up1(h))  # 2x upsample
        h = F.relu(self.up2(h))  # 4x total
        h = F.relu(self.up3(h))  # 8x total
        h = self.norm(h)
        return torch.sigmoid(self.conv_out(h))


vae_decoder = VideoVAEDecoder(
    out_channels=3, latent_channels=4, base_channels=32
).to(device)

with torch.no_grad():
    generated_video = vae_decoder(generated_latent)
print('Decoded video:', tuple(generated_video.shape))

fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, t in zip(axes, [0, 2, 5, 7]):
    img = generated_video[0].permute(1, 2, 3, 0)[t].cpu().numpy()
    ax.imshow(img)
    ax.set_title(f'frame {t}')
    ax.axis('off')
plt.suptitle('Decoded output (untrained VAE, so expect texture, not a movie)')
plt.tight_layout()
plt.show()

## Summary

You implemented every core component of a Latte-style video generation model:

1. **VAE compression** shrinks video into an efficient latent space
2. **Video tokenization and patch embedding** turn latents into position-aware token sequences
3. **Spatial and temporal attention**, alternated, capture structure within and across frames at tractable cost
4. **Diffusion training** teaches the model to reverse noise corruption
5. **Text conditioning** via T5 embeddings, cross-attention, and classifier-free guidance turns a random generator into a controllable one

Chapter 4 covers what comes next: training these models on real video datasets with the optimization and scaling techniques production systems rely on.

The reusable versions of these classes live in `chapter-03/src/`. From the book *Video Generation with AI* by Joseph Enochs (O'Reilly, Print ISBN 9798341653344).